# YOLO26 DFL-free

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/16-dfl-free/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Continuous distance regression removes bin softmax and its finite-bin range."""
import torch
from torch import nn
from torch.nn import functional as F


def decode(points, distances, stride):
    return torch.cat((points - distances[..., :2] * stride,
                      points + distances[..., 2:] * stride), -1)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    target = torch.tensor([[1.25, 2.5, 18., 3.]])  # feature-cell units
    distance = nn.Parameter(torch.zeros(1, 4))
    optimizer = torch.optim.SGD([distance], lr=.5)
    initial_terms = F.smooth_l1_loss(distance, target, reduction='none')
    assert torch.allclose(initial_terms, torch.tensor([[.75, 2., 17.5, 2.5]]))
    before = initial_terms.mean().item()
    for step in range(300):
        optimizer.zero_grad()
        loss = F.smooth_l1_loss(distance, target)
        loss.backward()
        if step == 0:
            assert torch.allclose(distance.grad, torch.full_like(distance, -.25))
            print('first Smooth L1 gradient:', distance.grad.tolist())
        optimizer.step()
        if step == 0:
            assert torch.allclose(distance, torch.full_like(distance, .125))
            print('first SGD distance:', distance.detach().tolist())
    after = F.smooth_l1_loss(distance, target).item()
    assert after < 1e-6
    k = 16
    logits = torch.zeros(1, 4, k)
    probabilities = logits.softmax(-1)
    expected = (probabilities * torch.arange(k)).sum(-1)
    assert torch.allclose(probabilities, torch.full_like(probabilities, 1 / 16))
    assert torch.allclose(expected, torch.full_like(expected, 7.5))
    assert expected.max() <= k - 1 and target.max() > k - 1
    dfl_target = torch.zeros(k)
    dfl_target[1], dfl_target[2] = .75, .25
    assert torch.allclose((dfl_target * torch.arange(k)).sum(), target[0, 0])
    # Candidate point: center of feature cell column 10, row 10, i.e. ((10+.5)*8, (10+.5)*8).
    points = torch.tensor([[84., 84.]])  # pixel x,y
    decoded = decode(points, distance.detach(), stride=8)
    assert torch.allclose(decoded, torch.tensor([[74., 64., 228., 108.]]), atol=1e-3)
    signed_box = decode(points, torch.tensor([[-1., 2., 3., 4.]]), stride=8)
    assert torch.allclose(signed_box, torch.tensor([[92., 68., 108., 116.]]))
    assert (signed_box[..., 2:] > signed_box[..., :2]).all()
    b, p, classes = 1, 100, 2
    raw_dfl = torch.zeros(b, p, 4 * k + classes)
    raw_direct = torch.zeros(b, p, 4 + classes)
    print('direct learned distances:', distance.detach().round(decimals=3).tolist())
    print(f'Smooth L1 {before:.6f} -> {after:.6f}')
    print('decoded direct box pixels:', decoded.round(decimals=3).tolist())
    print('signed-distance example box pixels:', signed_box.tolist())
    print('untrained DFL probability per bin:', probabilities[0, 0].tolist())
    print('untrained DFL expected distances:', expected.tolist())
    print('DFL target for 1.25: bins 1/2 weights .75/.25; direct target is the value 1.25')
    print('finite-bin expectation range:', [0, k - 1], '; direct target reaches', target.max().item())
    print('raw head values DFL / direct:', raw_dfl.numel(), raw_direct.numel())
    print('float32 raw head bytes:', raw_dfl.numel() * 4, raw_direct.numel() * 4)
    print('DFL-free still requires box supervision; this experiment uses Smooth L1')


if __name__ == '__main__':
    main()


first Smooth L1 gradient: [[-0.25, -0.25, -0.25, -0.25]]
first SGD distance: [[0.125, 0.125, 0.125, 0.125]]
direct learned distances: [[1.25, 2.5, 18.0, 3.0]]
Smooth L1 5.687500 -> 0.000000
decoded direct box pixels: [[74.0, 64.0, 228.0, 108.0]]
signed-distance example box pixels: [[92.0, 68.0, 108.0, 116.0]]
untrained DFL probability per bin: [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625]
untrained DFL expected distances: [[7.5, 7.5, 7.5, 7.5]]
DFL target for 1.25: bins 1/2 weights .75/.25; direct target is the value 1.25
finite-bin expectation range: [0, 15] ; direct target reaches 18.0
raw head values DFL / direct: 6600 600
float32 raw head bytes: 26400 2400
DFL-free still requires box supervision; this experiment uses Smooth L1
